In [1]:
import math
import numpy as np
import pandas as pd
from scipy.stats import norm

In [10]:
import pandas as pd

df = pd.read_csv(
    r"D:\FRM2\outputs\byd_ifind_20261027\002594_SZ_close_af_20211101_20261026.csv",
    encoding="utf-8-sig",
    parse_dates=["trade_date"],  # 将日期列转为日期类型
    dtype={"thscode": str},      # 股票代码按字符串读取
)

df = df.sort_values("trade_date").reset_index(drop=True)
df["modified_price"] = df['ths_close_price_stock'] * df['ths_af_stock'] #添加复权因子调整后的价格
df["position"] = 10000 #添加头寸
rng = np.random.default_rng(42)
df["interim_payments"] = rng.integers(-10000,10001, size = len(df))#添加持有期间发生的现金支付或收入
df["book_value"] = df['modified_price'] * df['position']
df["discount_rate"] = 0.02
print(df.head())   # 查看前 5 行
print(df.dtypes)   # 查看各列类型
print(df.shape)    # 应为 (1208, 4)

  trade_date    thscode  ths_close_price_stock  ths_af_stock  modified_price  \
0 2021-11-01  002594.SZ                 302.00      1.020391      308.158025   
1 2021-11-02  002594.SZ                 313.88      1.020391      320.280268   
2 2021-11-03  002594.SZ                 303.01      1.020391      309.188620   
3 2021-11-04  002594.SZ                 325.00      1.020391      331.627013   
4 2021-11-05  002594.SZ                 314.00      1.020391      320.402715   

   position  interim_payments    book_value  discount_rate  
0     10000             -8215  3.081580e+06           0.02  
1     10000              5479  3.202803e+06           0.02  
2     10000              3092  3.091886e+06           0.02  
3     10000             -1222  3.316270e+06           0.02  
4     10000             -1340  3.204027e+06           0.02  
trade_date               datetime64[us]
thscode                             str
ths_close_price_stock           float64
ths_af_stock                    f

# 1.1 DATA

In [ ]:
def select_base_date(df, date):
    result = df.copy()
    result["trade_date"] = pd.to_datetime(
        result["trade_date"]
    ).dt.normalize()
    result = result.sort_values("trade_date").reset_index(drop=True)

    if result["trade_date"].duplicated().any():
        raise ValueError("每个交易日只能有一条记录，请先筛选单只股票")

    base_date = pd.Timestamp(date).normalize()
    matches = result.index[result["trade_date"].eq(base_date)]

    if len(matches) != 1:
        raise ValueError("基准日期必须在数据中存在")

    result["discount_days"] = result.index - matches[0]
    return result

def cal_pnl(df, base_date):
    result = select_base_date(df, base_date)
    future = result.loc[result["discount_days"] >= 0].copy()

    if future[["book_value", "interim_payments"]].isna().any().any():
        raise ValueError("市值和现金流不能缺失")

    base_value = future["book_value"].iloc[0]

    payments = future["interim_payments"].copy()
    payments.iloc[0] = 0.0  # 不计入基准日当天现金流

    result["pnl"] = (
        future["book_value"]
        + payments.cumsum()
        - base_value
    )

    return result


def cal_pvpnl(df, base_date):
    result = select_base_date(df, base_date)
    future = result.loc[result["discount_days"] >= 0].copy()

    required = ["book_value", "interim_payments", "discount_rate"]
    if future[required].isna().any().any():
        raise ValueError("市值、现金流和折现率不能缺失")

    base_value = future["book_value"].iloc[0]
    daily_rate = future["discount_rate"] / 252

    growth = 1 + daily_rate
    growth.iloc[0] = 1.0
    accumulation = growth.cumprod()

    payments = future["interim_payments"].copy()
    payments.iloc[0] = 0.0

    # 各笔现金流分别折现到基准日，再累计
    pv_payments = (payments / accumulation).cumsum()

    result["pvpnl"] = (
        future["book_value"] / accumulation
        + pv_payments
        - base_value
    )

    return result


def cal_fvpnl(df, base_date):
    result = select_base_date(df, base_date)

    # 基准日作为终点，取此前的数据
    past = result.loc[result["discount_days"] <= 0].copy()

    required = ["book_value", "interim_payments", "discount_rate"]
    if past[required].isna().any().any():
        raise ValueError("市值、现金流和折现率不能缺失")

    base_value = past["book_value"].iloc[-1]
    daily_rate = past["discount_rate"] / 252

    # 每行利率用于上一交易日至当前交易日
    growth = 1 + daily_rate
    growth.iloc[0] = 1.0
    accumulation = growth.cumprod()

    # 每个过去日期到基准日的增长倍数
    to_base = accumulation.iloc[-1] / accumulation

    # 每笔现金流累积到基准日
    payments_at_base = past["interim_payments"] * to_base

    # 累计各起点之后的现金流，不包含起点当天
    subsequent_payments = (
        payments_at_base.iloc[::-1].cumsum().iloc[::-1]
        - payments_at_base
    )

    result["fvpnl"] = (
        base_value
        + subsequent_payments
        - past["book_value"] * to_base
    )

    return result

# arithmetic return
def ar(Pt1,Pt2,Dt):
    return [round((p2 + dt)/p1 -1, 2) for p1, p2, dt in zip(Pt1, Pt2, Dt)]

# geometric return
def gr(Pt1,Pt2,Dt):
    return [round(math.log((p2 + dt)/p1), 2) for p1, p2, dt in zip(Pt1, Pt2, Dt)]

# 1.2 Estimating Historical Simulation & parametric VaR

In [5]:
def _resolve_returns(ret, loss, V0):
    """由 ret（收益率）或 loss（损失，正=亏）得到【算术单期收益率】"""
    if ret is not None:
        return np.asarray(ret, dtype=float)
    if loss is not None:
        return -np.asarray(loss, dtype=float) / V0
    raise ValueError("必须提供 ret（收益率序列）或 loss（损失序列）之一")

def _to_log(r, log_returns):
    """转成对数收益率"""
    return r if log_returns else np.log1p(r)

def _agg(r_simple, T):
    """单期算术收益 -> T 期算术收益（滚动复合）"""
    if T == 1:
        return r_simple
    n = len(r_simple) - T + 1
    return np.array([np.prod(1.0 + r_simple[i:i + T]) - 1.0 for i in range(n)])

def _moments(r, mu=None, sigma=None):
    """估计单期收益的均值/标准差；显式给了就用给的"""
    m = float(r.mean())     if mu    is None else float(mu)
    s = float(r.std(ddof=1)) if sigma is None else float(sigma)
    return m, s

ALIAS = {
    "hs": "hs", "historical": "hs", "empirical": "hs", "历史模拟": "hs",
    "normal": "normal", "gaussian": "normal", "正态": "normal",
    "lognormal": "lognormal", "log-normal": "lognormal", "对数正态": "lognormal",
}

# ================= 统一入口（单方法） =================
def var(method="hs", *, ret=None, loss=None, V0=1.0, c=0.99, T=1,
        mu=None, sigma=None, log_returns=False):
    """
    三种 VaR 方法的统一入口。

    method      'hs'（历史模拟） / 'normal'（正态） / 'lognormal'（对数正态）
    ret         单期收益率序列；log_returns=True 表示传的是对数收益
    loss        单期损失序列（正=亏损），与 ret 二选一
    V0          组合当前市值（默认 1，此时返回的是相对 VaR）
    c           置信水平，0.95 / 0.99  （三方法统一）
    T           持有期（期数），多期按 sqrt(T) 或滚动复合处理
    mu, sigma   单期收益的均值/标准差；不给则从数据估计（ddof=1）

    返回 dict：VaR, ES, VaR%, ES%（正数 = 损失）
    """
    key = ALIAS.get(str(method).lower())
    if key is None:
        raise ValueError("method 必须是 'hs' / 'normal' / 'lognormal'")

    rs = _resolve_returns(ret, loss, V0)     # 算术单期收益
    rl = _to_log(rs, log_returns)            # 对数单期收益
    zc = norm.ppf(c)
    out = {"method": key, "c": c, "T": T}

    # ---- 1. 历史模拟法 ----
    if key == "hs":
        x = np.sort(-V0 * _agg(rs, T))[::-1]      # 损失，从大到小
        m = int(np.ceil(len(x) * (1 - c)))        # 尾部样本数
        out.update({"VaR": x[m - 1], "ES": x[:m].mean(),
                    "n": len(x), "tail_n": m})

    # ---- 2. 正态（算术收益） ----
    elif key == "normal":
        mu1, sd1 = _moments(rs, mu, sigma)
        muT, sdT = mu1 * T, sd1 * np.sqrt(T)
        out.update({
            "mu_T": muT, "sigma_T": sdT,
            "VaR": V0 * (zc * sdT - muT),
            "ES":  V0 * (sdT * norm.pdf(zc) / (1 - c) - muT),
        })

    # ---- 3. 对数正态（几何收益） ----
    else:
        mu1, sd1 = _moments(rl, mu, sigma)
        muT, sdT = mu1 * T, sd1 * np.sqrt(T)
        out.update({
            "mu_T": muT, "sigma_T": sdT,
            "VaR": V0 * (1.0 - np.exp(muT - zc * sdT)),
            "ES":  V0 * (1.0 - np.exp(muT + 0.5 * sdT**2)
                         * norm.cdf(-zc - sdT) / (1 - c)),
        })

    out["VaR%"] = out["VaR"] / V0
    out["ES%"]  = out["ES"]  / V0
    return out


# ================= 统一入口（三方法一次性对比） =================
def var_all(*, ret=None, loss=None, V0=1.0, levels=(0.95, 0.99), T=1,
            mu=None, sigma=None, log_returns=False):
    """一次性跑三种方法 × 多个置信度，返回对比表"""
    rows = []
    for c in np.atleast_1d(levels):
        for m in ("hs", "normal", "lognormal"):
            r = var(m, ret=ret, loss=loss, V0=V0, c=float(c), T=T,
                    mu=mu, sigma=sigma, log_returns=log_returns)
            rows.append({"置信度": f"{c:.1%}", "方法": m,
                         "VaR": round(r["VaR"], 2), "ES": round(r["ES"], 2),
                         "VaR%": f"{r['VaR%']:.2%}", "ES%": f"{r['ES%']:.2%}"})
    return pd.DataFrame(rows).set_index(["置信度", "方法"])

In [9]:
# ---------- 参数 ----------
N_OBS = 2000          # 观测个数
V0    = 1_000_000     # 组合当前市值
MU_D  = 0.0004        # 日均收益率
SIG_D = 0.010         # 日波动率 1%
DF    = 5             # 学生 t 自由度（越小尾部越厚，更接近真实市场）
SEED  = 20261023      # 固定种子，结果可复现

rng = np.random.default_rng(SEED)

# ---------- 生成收益率 ----------
# A. 正态分布（简单，尾部偏薄）
r_norm = rng.normal(MU_D, SIG_D, N_OBS)

# B. 学生 t 分布（推荐：有厚尾，练 VaR 更真实）
#    t(df) 的方差是 df/(df-2)，除以 sqrt(df/(df-2)) 把标准差调到 SIG_D
r_t = MU_D + SIG_D * rng.standard_t(DF, N_OBS) / np.sqrt(DF / (DF - 2))

# ---------- 转成 P&L 与 Loss ----------
# 约定：loss 为正数表示亏损（历史模拟法的标准做法）
ret   = r_t
PL    = V0 * ret          # 正 = 盈利
loss  = -PL               # 正 = 亏损

# ---------- 装进 DataFrame，带交易日索引 ----------
idx = pd.bdate_range("2018-01-01", periods=N_OBS, name="date")
df  = pd.DataFrame({"return": ret, "PL": PL, "loss": loss}, index=idx)

print(df.head())
print(df.shape)           # (2000, 3)
print("亏损天数占比:", (df["loss"] > 0).mean().round(4))

V0 = 1_000_000
r_log = df["return"].to_numpy()        # 上一问生成的 2000 个对数收益

# 单个方法
var("hs",        ret=r_log, V0=V0, c=0.99, log_returns=True)
var("normal",    ret=r_log, V0=V0, c=0.99, log_returns=True)
var("lognormal", ret=r_log, V0=V0, c=0.99, log_returns=True)

# 三方法 × 两置信度 对比
var_all(ret=r_log, V0=V0, levels=(0.95, 0.99), log_returns=True)


              return            PL          loss
date                                            
2018-01-01  0.016043  16042.790576 -16042.790576
2018-01-02 -0.012737 -12736.631045  12736.631045
2018-01-03  0.003084   3083.610263  -3083.610263
2018-01-04  0.006263   6263.317137  -6263.317137
2018-01-05 -0.014348 -14348.188150  14348.188150
(2000, 3)
亏损天数占比: 0.4635


VaR        ES   VaR%    ES%
置信度   方法                                         
95.0% hs         15267.81  21940.73  1.53%  2.19%
      normal     15648.71  19717.30  1.56%  1.97%
      lognormal  15526.90  19517.78  1.55%  1.95%
99.0% hs         26285.95  33309.32  2.63%  3.33%
      normal     22284.24  25583.70  2.23%  2.56%
      lognormal  22037.79  25254.74  2.20%  2.53%